In [ ]:
import ipywidgets as widgets
from ipyleaflet import Map, DrawControl
from fetchez.registry import ModuleRegistry, BundleRegistry

shared_layout = widgets.Layout(height="auto")

# --- Header Block ---
header = widgets.HTML("<h2>🌐 Globato DEM Builder</h2><hr/>")

# --- Load Registries & Extract Descriptions ---
ModuleRegistry.load_all()
BundleRegistry.load_all()
registry = ModuleRegistry.get_registry()
registry.update(BundleRegistry.get_registry())

globato_sources = {}
for name, meta in sorted(registry.items()):
    # Filter for modules/bundles tagged with 'glob-stream'
    if "glob-stream" in meta.get("tags", []) and name not in meta.get("aliases", []):
        # Extract the description to use for our hover tooltip
        desc = meta.get("description") or meta.get("desc", "No description provided.")
        globato_sources[name] = desc.strip().split("\n")[0]

# --- Left Column: Map Selector ---
m = Map(center=[44.6, -124.05], zoom=10, layout=shared_layout)
draw_control = DrawControl(rectangle={"shapeOptions": {"color": "#0074D9"}})
m.add(draw_control)
m.add_raster(
    "/home/ncei/Projects/tmp/newport/newport_dem/newport_n44x64_w124x10_final.tif",
    colormap="terrain",
    layer_name="Newport DEM",
)

# --- Right Column: Form Options ---
# Group A: Processing Parameters
region_input = widgets.Text(description="Region:", placeholder="Auto-populated by map")
increment_input = widgets.Dropdown(
    options=["1s", "1/3s", "1/9s", "3s", "30m"], value="1s", description="Increment:"
)
srs_input = widgets.Dropdown(
    options=["EPSG:4326", "EPSG:4269+5703", "EPSG:3857"],
    value="EPSG:4326",
    description="Target SRS:",
)

# Group B: Output & Storage Settings
outname_input = widgets.Text(
    description="Output Name:", value="globato_dem", placeholder="e.g., newport_dem"
)
outdir_input = widgets.Text(
    description="Out Folder:",
    value="~/workshop/output",
    placeholder="Output directory path",
)
cache_input = widgets.Text(
    description="Cache Dir:",
    value="~/workshop/shared_cache",
    placeholder="Shared cache path",
)

# Group C: Dynamic Sources with Hover Tooltips
source_checkboxes = []
for src_name, src_desc in globato_sources.items():
    cb = widgets.Checkbox(
        value=False,
        description=src_name,
        indent=False,
        tooltip=src_desc,  # <-- Native ipywidgets hover text!
    )
    source_checkboxes.append(cb)

sources_ui = widgets.VBox(
    [widgets.HTML("<b>Data Sources (Hover for info):</b>")] + source_checkboxes,
    layout=widgets.Layout(
        max_height="160px", overflow="auto", border="1px solid #ddd", padding="5px"
    ),
)

# Group D: Build Execution
build_button = widgets.Button(
    description="Build DEM",
    button_style="success",
    icon="play",
    layout=widgets.Layout(margin="10px 0px 0px 0px", width="100%"),
)
output_log = widgets.Output()

clear_button = widgets.Button(
    description="Clear Log",
    button_style="info",
    icon="refresh",
    layout=widgets.Layout(margin="10px 0px 0px 0px", width="100%"),
)

# Assemble the Right Column
form_ui = widgets.VBox(
    [
        widgets.HTML("<b>1. Spatial Configuration</b>"),
        region_input,
        increment_input,
        srs_input,
        widgets.HTML("<hr/><b>2. Storage & Outputs</b>"),
        outname_input,
        outdir_input,
        cache_input,
        widgets.HTML("<hr/>"),
        sources_ui,
        widgets.HTML("<hr/>"),
        build_button,
        clear_button,
    ]
)


# --- Event Handlers ---
def on_draw(target, action, geo_json):
    if action == "created":
        coords = geo_json["geometry"]["coordinates"][0]
        lons = [p[0] for p in coords]
        lats = [p[1] for p in coords]
        region_input.value = (
            f"{min(lons):.5f}/{max(lons):.5f}/{min(lats):.5f}/{max(lats):.5f}"
        )


def on_build_clicked(b):
    with output_log:
        output_log.clear_output()

        selected_sources = [cb.description for cb in source_checkboxes if cb.value]

        if not selected_sources:
            print("⚠️ Please select at least one data source!")
            return

        print(f"🚀 Triggering Globato build for {region_input.value}...")
        print(f"📦 Sources: {', '.join(selected_sources)}")
        print(f"📁 Saving '{outname_input.value}' to {outdir_input.value}")
        print(f"🗄️ Using Shared Cache: {cache_input.value}")

        # globato.api.build(
        #     region=region_input.value,
        #     increment=increment_input.value,
        #     t_srs=srs_input.value,
        #     outname=outname_input.value,
        #     outdir=outdir_input.value,
        #     shared_cache=cache_input.value,
        #     sources=selected_sources
        # )


def on_clear_clicked(b):
    with output_log:
        output_log.clear_output()


draw_control.on_draw(on_draw)
build_button.on_click(on_build_clicked)
clear_button.on_click(on_clear_clicked)

# --- Assemble Dashboard ---
dashboard = widgets.VBox(
    [header, widgets.HBox([m, form_ui]), widgets.HTML("<hr/>"), output_log]
)


display(dashboard)